<!--
    <div style='text-align: center;'>
        <h1>LTX-2.3 22B Distilled 1.1 Q4 - Video Generator</h1>
        <h3>Kaggle T4 GPU Edition</h3>
    </div>
-->

NOTE: After running Cell 1, restart kernel (Kernel -> Restart Kernel) and run all cells again.

In [ ]:
# Install PyTorch FIRST before any other imports
import os
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'

%pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
print('PyTorch 2.3.1 installed. Now restart kernel and run all cells.')

### Step 2: Install Dependencies

In [ ]:
import os, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

%pip install -q mmgp gradio gguf soundfile pillow tqdm einops omegaconf ffmpeg-python
!git clone --depth 1 https://github.com/DeepBeepMeep/Wan2GP.git && echo 'Wan2GP cloned'

### Step 3: Download Models

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

MODEL_DIR = '/kaggle/working/models'
TMP_DIR = '/kaggle/tmp/models'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# Copy models from dataset
dataset_path = '/kaggle/input/datasets/investorblak/ltx-batch'
if os.path.exists(dataset_path):
    for f in os.listdir(dataset_path):
        if f.endswith('.safetensors') or f.endswith('.gguf'):
            shutil.copy(os.path.join(dataset_path, f), os.path.join(MODEL_DIR, f))
            print(f'Copied {f}')

# Download Gemma if missing
gemma_folder = 'gemma-3-12b-it-qat-q4_0-unquantized'
gemma_dir = os.path.join(MODEL_DIR, gemma_folder)
gemma_tmp = os.path.join(TMP_DIR, gemma_folder)
if not os.path.isdir(gemma_dir) or not os.listdir(gemma_dir):
    print('Downloading Gemma text encoder...')
    os.makedirs(gemma_tmp, exist_ok=True)
    for gf in ['gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors',
               'tokenizer.json', 'tokenizer.model',
               'config_light.json', 'generation_config.json',
               'preprocessor_config.json', 'processor_config.json',
               'special_tokens_map.json', 'chat_template.json',
               'tokenizer_config.json', 'added_tokens.json']:
        try:
            hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename=f'{gemma_folder}/{gf}', local_dir=TMP_DIR)
        except: pass
    os.symlink(gemma_tmp, gemma_dir)

print('All models ready')

### Step 4: Copy Script

In [ ]:
import shutil
script_src = '/kaggle/input/datasets/investorblak/ltx-batch/run_ltx_t2v.py'
if os.path.exists(script_src):
    shutil.copy(script_src, '/kaggle/working/run_ltx_t2v.py')
    print('Copied run_ltx_t2v.py from dataset')
else:
    print('WARNING: Script not in dataset')

### Step 5: Launch

In [ ]:
print('\nLaunching LTX Video Generator...')
print('Output: /kaggle/working/outputs/ltx_video_0001.mp4, etc.')
!cd /kaggle/working && python run_ltx_t2v.py